# Comprehensive EDA: Before & After Cleaning (Google Colab Version)

Explore the static NYC Taxi Parquet sample, clean it, and visually compare the data before and after cleaning.

In [ ]:
# Install dependencies for Google Colab
!pip install pyspark seaborn matplotlib pandas -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, isnull, hour
from pyspark.sql.types import IntegerType, FloatType, TimestampType
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

spark = SparkSession.builder \
    .appName("ComprehensiveEDA") \
    .master("local[*]") \
    .getOrCreate()
print("Spark Session is Ready!")

### 1. Load Data & Schema Transform

In [ ]:
raw_df = spark.read.parquet("yellow_tripdata_2025-01_sample_10000.parquet")

df_before = raw_df \
    .withColumn("pickup_datetime", col("tpep_pickup_datetime").cast(TimestampType())) \
    .withColumn("dropoff_datetime", col("tpep_dropoff_datetime").cast(TimestampType())) \
    .withColumn("passenger_count", col("passenger_count").cast(IntegerType())) \
    .withColumn("trip_distance", col("trip_distance").cast(FloatType())) \
    .withColumn("pickup_location_id", col("PULocationID").cast(IntegerType())) \
    .withColumn("dropoff_location_id", col("DOLocationID").cast(IntegerType())) \
    .withColumn("fare_amount", col("fare_amount").cast(FloatType())) \
    .withColumn("tip_amount", col("tip_amount").cast(FloatType())) \
    .withColumn("total_amount", col("total_amount").cast(FloatType())) \
    .withColumn("payment_type", col("payment_type").cast(IntegerType())) \
    .select(
        "pickup_datetime", "dropoff_datetime", "passenger_count", 
        "trip_distance", "pickup_location_id", "dropoff_location_id", 
        "fare_amount", "tip_amount", "total_amount", "payment_type"
    )


### 2. Advanced Logical Integrity Checks (Edge Cases)

In [ ]:
print("--- Advanced Logical Integrity Checks ---")

# 1. Clown Car (Too many passengers)
clown_cars = df_before.filter(col("passenger_count") > 8).count()
print(f"Trips with > 8 passengers (Clown cars): {clown_cars}")

# 2. Invalid TLC Zones (NYC has 265 official taxi zones)
invalid_pu_zones = df_before.filter(~col("pickup_location_id").between(1, 265)).count()
invalid_do_zones = df_before.filter(~col("dropoff_location_id").between(1, 265)).count()
print(f"Trips with invalid Pickup Zones: {invalid_pu_zones}")
print(f"Trips with invalid Dropoff Zones: {invalid_do_zones}")

# 3. Extreme Fare Outliers (Billionaire trips)
extreme_fares = df_before.filter(col("fare_amount") > 1000).count()
print(f"Trips with a base fare over $1,000: {extreme_fares}")

# 4. Invalid Payment Types (Officially 1-6)
invalid_payments = df_before.filter(~col("payment_type").between(1, 6)).count()
print(f"Trips with unknown/invalid payment types: {invalid_payments}")


### 3. Ultimate Data Cleaning Pipeline

In [ ]:
# Apply ULTRA-STRICT cleaning pipeline 
df_after = df_before \
    .fillna(0, subset=["passenger_count"]) \
    .dropDuplicates() \
    .dropna(subset=["pickup_datetime", "dropoff_datetime"]) \
    .filter(
        (col("fare_amount") >= 0) &                                             # Realistic fare bounds (no negatives)
        (col("passenger_count") <= 8) &                                         # No clown cars
        (col("dropoff_datetime") > col("pickup_datetime")) &                    # No time travel
        ~((col("trip_distance") == 0) & (col("fare_amount") > 10)) &            # No GPS glitches
        col("pickup_location_id").between(1, 265) &                             # Valid PU Zone
        col("dropoff_location_id").between(1, 265) &                            # Valid DO Zone
        col("payment_type").between(0, 6)                                       # Valid Payment (Allow 0 since they correlate with missing passenger counts!)
    )

print(f"Total Rows BEFORE cleaning: {df_before.count()}")
print(f"Total Rows AFTER ultra-strict cleaning: {df_after.count()}")
print(f"Rows dropped: {df_before.count() - df_after.count()}")

### 4. Visualization Function

In [ ]:
def plot_dashboard(spark_df, title):
    pdf = spark_df.withColumn("pickup_hour", hour("pickup_datetime")).toPandas()
    
    payment_map = {
        0: "0 (Unmapped)", 1: "1 (Credit)", 2: "2 (Cash)", 
        3: "3 (No Charge)", 4: "4 (Dispute)", 5: "5 (Unknown)", 
        6: "6 (Voided)"
    }
    pdf["payment_type_label"] = pdf["payment_type"].map(payment_map).fillna(pdf["payment_type"].astype(str))
    
    fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(20, 18))
    fig.suptitle(title, fontsize=24, y=1.02, fontweight='bold')
    
    sns.countplot(data=pdf, x="pickup_hour", ax=axes[0, 0], palette="crest")
    axes[0, 0].set_title("1. Pickup Hours (Time of Day)", fontsize=14)
    
    sns.countplot(data=pdf, x="passenger_count", ax=axes[0, 1], palette="viridis")
    axes[0, 1].set_title("2. Passenger Count Distribution", fontsize=14)
    
    sns.countplot(data=pdf, y="payment_type_label", ax=axes[0, 2], palette="magma", order=sorted(pdf["payment_type_label"].dropna().unique()))
    axes[0, 2].set_title("3. Payment Type", fontsize=14)
    
    sns.histplot(pdf["trip_distance"], bins=40, kde=False, ax=axes[1, 0], color="dodgerblue")
    axes[1, 0].set_title("4. Trip Distance (All Values)", fontsize=14)
    axes[1, 0].set_yscale("log")
    
    top_pu = pdf["pickup_location_id"].value_counts().nlargest(10).reset_index()
    sns.barplot(data=top_pu, x="pickup_location_id", y="count", ax=axes[1, 1], palette="rocket")
    axes[1, 1].set_title("5. Top 10 Pickup Locations", fontsize=14)
    
    top_do = pdf["dropoff_location_id"].value_counts().nlargest(10).reset_index()
    sns.barplot(data=top_do, x="dropoff_location_id", y="count", ax=axes[1, 2], palette="mako")
    axes[1, 2].set_title("6. Top 10 Dropoff Locations", fontsize=14)
    
    sns.histplot(pdf["fare_amount"], bins=50, kde=False, ax=axes[2, 0], color="green")
    axes[2, 0].set_title("7. Fare Amount (All Values)", fontsize=14)
    axes[2, 0].set_yscale("log")
    
    sns.histplot(pdf["tip_amount"], bins=30, kde=False, ax=axes[2, 1], color="orange")
    axes[2, 1].set_title("8. Tip Amount", fontsize=14)
    axes[2, 1].set_yscale("log")
    
    sns.scatterplot(data=pdf, x="trip_distance", y="total_amount", ax=axes[2, 2], alpha=0.3, color="purple")
    axes[2, 2].set_title("9. Total Fare vs Distance", fontsize=14)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_dashboard(df_before, "🚨 BEFORE CLEANING: Raw NYC Taxi Data")

In [ ]:
plot_dashboard(df_after, "✅ AFTER CLEANING: Cleaned & Filtered Data")